# Protein interface analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davismerighi-max/interface-analysis/blob/main/notebooks/interface_analysis_colab.ipynb)

Give it a refined structure (`.pdb` / `.cif`, or a PDB entry code) and it measures every
interface: which residues are buried and by how much, the salt bridges, hydrogen bonds,
hydrophobic contacts, disulfides, metal bridges and stacking that hold the subunits
together, how well the surfaces fit, what bound cofactors touch, and which parts of the
model could be stable assemblies on their own. Then it draws the figures and hands you
everything as a zip.

**No programming needed.** Run the cells from top to bottom (`Shift`+`Enter`, or ▶ on each
cell). Boxes **2**, **3** and **3b** show their fields and upload buttons *after* you run them:
fill them in, then go on to the next box. Settings shown as form fields (resolution, residues
per line, …) are set before running their box. *Runtime → Run all* works too and analyses the
example entry 2TSC. Symmetry, chain count and the number of
different proteins are measured from your model, never declared.

Code and documentation: https://github.com/davismerighi-max/interface-analysis

## 1 · Set up

Installs the libraries and downloads the code. About a minute; once per session.


In [ ]:
#@title Install and fetch the code { display-mode: "form" }
import os, sys, subprocess, shutil, pathlib
print("installing libraries ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "biopython", "pydssp", "scikit-image"], check=True)
REPO = "/content/interface-analysis"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "-q",
                "https://github.com/davismerighi-max/interface-analysis.git", REPO], check=True)
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
for f in pathlib.Path(REPO).glob("*.py"):
    shutil.copy(f, WORK)
os.chdir(WORK)
print("ready — working directory is", os.getcwd())

## 2 · Give it a structure

Run this box, then **either** type a PDB entry code **or** click **Upload** and pick your own
`.pdb` / `.cif` file. Whichever you do last is the one used — the line under the boxes always
says which. Very large files (over ~20 MB) upload more reliably through the **Files** panel on
the left: drop the file there and type its name in the PDB-code box instead.

In [ ]:
#@title Load the model { display-mode: "form" }
import os, urllib.request, ipywidgets as W
from IPython.display import display

def _uploaded(w):
    """[(name, bytes)] from a FileUpload widget, for ipywidgets 7 and 8 alike."""
    v = w.value
    if isinstance(v, dict):
        return [(n, bytes(d["content"])) for n, d in v.items()]
    return [(d["name"], bytes(d["content"])) for d in v]

def _reset(w):
    """Empty a FileUpload once its file is taken, so picking the same file again still
    registers as a new upload (ipywidgets 7 and 8)."""
    try:
        w.value = {} if isinstance(w.value, dict) else ()
        if hasattr(w, "_counter"):
            w._counter = 0
    except Exception:
        pass

_code = W.Text(value="2TSC", placeholder="e.g. 2TSC", description="PDB code",
               layout=W.Layout(width="230px"))
_up = W.FileUpload(accept=".pdb,.cif,.ent,.mmcif", multiple=False,
                   description="Upload my file", layout=W.Layout(width="190px"))
_or = W.HTML("<span style='margin:0 14px;color:#777'>or</span>")
_status = W.HTML()
SOURCE = {"kind": "code"}

def _show():
    c = _code.value.strip()
    if SOURCE["kind"] == "file":
        _status.value = (f"✔ will use your uploaded file <b>{SOURCE['name']}</b> "
                         "(the PDB-code box is ignored)")
    elif c.lower().endswith((".pdb", ".cif", ".ent", ".mmcif")):
        _status.value = f"✔ will use the file <b>{c}</b> from the Files panel"
    elif c:
        _status.value = f"✔ will use PDB entry <b>{c.upper()}</b> (any uploaded file is ignored)"
    else:
        _status.value = "<span style='color:#B33A3A'>type a PDB code or upload a file</span>"

def _on_code(ch):
    if ch["new"].strip():
        SOURCE.clear(); SOURCE["kind"] = "code"
    _show()

def _on_upload(ch):
    files = _uploaded(_up)
    if files:
        name, data = files[-1]
        SOURCE.clear(); SOURCE.update(kind="file", name=name, data=data)
        _code.unobserve(_on_code, "value"); _code.value = ""; _code.observe(_on_code, "value")
        _reset(_up)
    _show()

_code.observe(_on_code, "value"); _up.observe(_on_upload, "value")
_show()
display(W.VBox([W.HBox([_code, _or, _up]), _status]))

def load_model():
    """Write or download the chosen structure; called by step 1."""
    global MODEL
    if SOURCE.get("kind") == "file":
        MODEL = os.path.basename(SOURCE["name"]).replace(" ", "_")
        open(MODEL, "wb").write(SOURCE["data"])
    else:
        c = _code.value.strip()
        if not c:
            raise SystemExit("Box 2: type a PDB code or upload a file, then run step 1 again.")
        if c.lower().endswith((".pdb", ".cif", ".ent", ".mmcif")):   # a file from the Files panel
            src = c if os.path.exists(c) else os.path.join("/content", c)
            if not os.path.exists(src):
                raise SystemExit(f"Box 2: no file called {c} - check the Files panel.")
            MODEL = os.path.basename(src)
            if os.path.abspath(src) != os.path.abspath(MODEL):
                shutil.copy(src, MODEL)
        else:
            code_ = c.lower()
            for ext in (".pdb", ".cif"):        # very large entries exist only as mmCIF
                try:
                    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{code_}{ext}",
                                               code_ + ext)
                    MODEL = code_ + ext
                    break
                except Exception:
                    continue
            else:
                raise SystemExit(f"Box 2: could not download PDB entry {c.upper()}.")
    n = sum(1 for l in open(MODEL, errors="ignore") if l.startswith(("ATOM", "HETATM")))
    print("model:", MODEL, "-", n, "atom records")
    return MODEL

## 3 · Name and colour your proteins (optional)

One line per protein: **name**, **colour** (click the swatch, or type `#RRGGBB`) and
**sequence**. Click **add a protein** for more lines. Proteins are recognised in the model **by
sequence**, so the order does not matter and the full-length sequence is fine even if your
model misses loops or termini. Leave everything empty to use the defaults. Anything that does
not match runs with defaults, and the run report says which and why.

Or upload a FASTA file whose headers carry the name and colour (`>YfdQ_#7FA8D4`); it replaces
the lines above. Keep names short — they label every axis.

In [ ]:
#@title Proteins { display-mode: "form" }
_PAL = ["#7FA8D4", "#E8884A", "#95C892", "#B48BC4", "#D4B36A", "#7FC4C4"]
_rows = W.VBox()

def _new_row(_=None):
    k = len(_rows.children)
    name = W.Text(placeholder="name", layout=W.Layout(width="150px"))
    col = W.ColorPicker(value=_PAL[k % len(_PAL)], layout=W.Layout(width="150px"))
    seq = W.Text(placeholder="sequence (one-letter code, paste the whole chain)",
                 layout=W.Layout(flex="1 1 auto", min_width="300px"))
    rm = W.Button(icon="times", tooltip="remove this protein", layout=W.Layout(width="38px"))
    row = W.HBox([name, col, seq, rm], layout=W.Layout(width="100%"))
    def _rm(_):
        if len(_rows.children) > 1:
            _rows.children = tuple(r for r in _rows.children if r is not row)
        else:
            name.value = ""; seq.value = ""
    rm.on_click(_rm)
    _rows.children = _rows.children + (row,)

_new_row()
_add = W.Button(description="add a protein", icon="plus", layout=W.Layout(width="150px"))
_add.on_click(_new_row)
_fasta = W.FileUpload(accept=".fasta,.fa,.faa,.txt", multiple=False,
                      description="or upload a FASTA", layout=W.Layout(width="190px"))
_fclear = W.Button(description="use the lines instead", layout=W.Layout(width="170px"))
_pstatus = W.HTML()
FASTA = {}

def _pshow():
    if FASTA:
        _pstatus.value = (f"✔ will use <b>{FASTA['name']}</b>; the lines above are ignored")
    else:
        _pstatus.value = "the lines above are used (empty lines are skipped)"

def _on_fasta(ch):
    f = _uploaded(_fasta)
    if f:
        FASTA.update(name=f[-1][0], data=f[-1][1])
        _reset(_fasta)
    _pshow()

def _on_fclear(_):
    FASTA.clear(); _pshow()

_fasta.observe(_on_fasta, "value"); _fclear.on_click(_on_fclear); _pshow()
_hdr = W.HTML("<div style='display:flex;color:#666;font-size:12px'>"
              "<span style='width:150px'>name</span><span style='width:150px'>colour</span>"
              "<span>sequence</span></div>")
display(W.VBox([_hdr, _rows, W.HBox([_add, _fasta, _fclear]), _pstatus]))

def write_proteins():
    """proteins.fasta from the lines or the uploaded FASTA (None if nothing given)."""
    if FASTA:
        open("proteins.fasta", "wb").write(FASTA["data"])
        return "proteins.fasta"
    out = []
    for i, row in enumerate(_rows.children, 1):
        name, col, seq = row.children[0].value, row.children[1].value, row.children[2].value
        name = name.strip().replace(" ", "_"); seq = "".join(seq.split())
        if name and seq:
            out.append(f">{name}_{col}\n{seq}\n")
        elif name or seq:
            print(f"protein line {i}: needs both a name and a sequence; skipped")
    if not out:
        return None
    open("proteins.fasta", "w").write("".join(out))
    return "proteins.fasta"

## 3b · Conservation (optional)

Upload one multiple sequence alignment per protein to add a **conservation track** to the
footprint figures — aligned FASTA, A3M (e.g. from ColabFold or HHblits), Clustal or Stockholm.
The **first** sequence must be your protein; each file is matched to its protein by that
sequence, so the file names do not matter. Without an alignment the footprints are drawn
without the track. Grades are entropy-based (Henikoff-weighted), not ConSurf.

In [ ]:
#@title Alignments { display-mode: "form" }
_msa = W.FileUpload(accept=".fasta,.fa,.a3m,.aln,.sto,.stockholm,.clustal,.txt", multiple=True,
                    description="Upload alignment(s)", layout=W.Layout(width="210px"))
_mclear = W.Button(description="remove alignments", layout=W.Layout(width="170px"))
_mstatus = W.HTML()
MSAS = {}

def _mshow():
    _mstatus.value = ("✔ " + ", ".join(f"<b>{n}</b>" for n in MSAS)) if MSAS else \
        "no alignment: footprints are drawn without a conservation track"

def _on_msa(ch):
    files = _uploaded(_msa)
    for n, d in files:
        MSAS[n] = d
    if files:
        _reset(_msa)
    _mshow()

def _on_mclear(_):
    MSAS.clear(); _mshow()

_msa.observe(_on_msa, "value"); _mclear.on_click(_on_mclear); _mshow()
display(W.VBox([W.HBox([_msa, _mclear]), _mstatus]))

def write_msas():
    """Save the alignments and return the --msa argument ('' if none)."""
    paths = []
    for i, (n, d) in enumerate(MSAS.items(), 1):
        safe = "".join(c if c.isalnum() or c in "._-" else "_" for c in n)
        p = f"msa_{i}_{safe}"
        open(p, "wb").write(d); paths.append(p)
    return ",".join(paths)

## 4 · Measure the interfaces

The main calculation: symmetry, contacts, buried surface atom by atom, the chemistry of every
contact and the shape complementarity of every interface. `resolution` is that of your map or
diffraction data (0 if unknown). Shape complementarity is the slow part — about a minute for a
dimer, ten minutes or more for 30 chains; untick it for a quick first look (the packing figure
is then left out).

In [ ]:
#@title Run step 1 { display-mode: "form" }
resolution = 1.97  #@param {type:"number"}
shape_complementarity = True  #@param {type:"boolean"}
MODEL = load_model()
PROTEINS = write_proteins()
print("protein details:", PROTEINS or "none given, defaults will be used")
cmd = f"python 01_interface_analysis.py {MODEL}"
if resolution and resolution > 0:
    cmd += f" --resolution {resolution:g}"
cmd += " --shape-workers 2" if shape_complementarity else " --no-shape"
if PROTEINS:
    cmd += f" --proteins {PROTEINS}"
print(cmd, "\n")
!{cmd}

## 5 · Tables and statistics

No settings: it reads them back from step 1.


In [ ]:
#@title Run step 2 and show the statistics { display-mode: "form" }
!python 02_derive_tables.py {MODEL}
print("\n" + "=" * 78 + "\n")
print(open("stats_summary.txt", encoding="utf-8").read())

## 6 · Figures

Besides the 2D figures, step 3 draws two quick 3D pictures of the whole model in the same
licorice style as the assembly renders: one coloured by the protein that buries each residue
(the *Interface budget*), one by the strongest interaction each residue makes (*Directional
interactions*). They are for a fast look; for rotating and close-ups use the ChimeraX files in
the download. Interfaces are mostly inside an assembly, so try `view_elevation = 0` for a
side view.


In [ ]:
#@title Run step 3 { display-mode: "form" }
residues_per_line = 350  #@param {type:"integer"}
draw_3D_renders = True  #@param {type:"boolean"}
view_elevation = 62  #@param {type:"number"}
view_azimuth = 20  #@param {type:"number"}
_render = (f"--render-view {view_elevation},{view_azimuth}" if draw_3D_renders
           else "--no-render")
_m = write_msas()
_msa_arg = f"--msa {_m}" if _m else ""
!python 03_make_figures.py {MODEL} --per-line {residues_per_line} {_render} {_msa_arg}

In [ ]:
#@title Show the figures here { display-mode: "form" }
import glob, base64, os
from IPython.display import Image, display, Markdown, HTML
renders = [f for f in ("fig_render_budget.png", "fig_render_interactions.png")
           if os.path.exists(f)]
figs = [f for f in sorted(glob.glob("fig*.png")) if f not in renders]
if not figs:
    print("no figures found - see the run report below")
if renders:
    display(Markdown("### 3D renders of the whole model"))
    imgs = "".join(
        f'<img src="data:image/png;base64,{base64.b64encode(open(f, "rb").read()).decode()}" '
        f'style="width:460px;margin-right:12px">' for f in renders)
    display(HTML(f'<div style="display:flex;flex-wrap:wrap">{imgs}</div>'))
for f in figs:
    display(Markdown("### " + f))
    display(Image(filename=f, width=950))


## 7 · Which parts could be stable assemblies?

Takes the model apart weakest interface first and classifies every piece that appears, with
the reason. **kept** = share of a piece's buried area that stays inside it:
≥ 67 % strong candidate · ≥ 50 % candidate · < 50 % not supported by contacts alone ·
*covalent unit* = chains joined by disulfides. These cut-offs are a stated heuristic, not a
published standard; disulfides are reported separately, never counted as area. Treat the
table as evidence to test (SEC-MALS, mass photometry, non-reducing gels), not as a verdict.


In [ ]:
#@title Run step 5 and show the candidates { display-mode: "form" }
draw_assemblies = True  #@param {type:"boolean"}
!python 05_assemblies.py {MODEL} {"--render" if draw_assemblies else ""} > /dev/null
import pandas as pd, os
from IPython.display import display, Image
pd.set_option("display.max_colwidth", 200)
if os.path.exists("assembly_candidates.csv"):
    t = pd.read_csv("assembly_candidates.csv")
    display(t[["rank", "class", "kept_pct", "n_chains", "composition",
               "disulfides_inside", "disulfides_across_boundary", "reason"]])
if draw_assemblies and os.path.exists("fig_assemblies.png"):
    display(Image(filename="fig_assemblies.png", width=950))

## 8 · Run report

What worked, what ran with defaults, and what failed — check this before trusting any figure.


In [ ]:
#@title Show the run report { display-mode: "form" }
print(open("run_report.txt", encoding="utf-8").read())
if os.path.exists("run_errors.log"):
    print("\nDetails of any failure are in run_errors.log (included in the zip).")

## 9 · Take everything with you

Tables are plain CSV; figures are PNG (600 dpi) and vector PDF; the ChimeraX scripts open your model in flat, silhouetted licorice.


In [ ]:
#@title Download a zip of all results { display-mode: "form" }
import shutil
base = os.path.splitext(os.path.basename(MODEL))[0] + "_interface_analysis"
shutil.make_archive("/content/" + base, "zip", "/content/work")
print("packed", base + ".zip")
try:
    from google.colab import files
    files.download("/content/" + base + ".zip")
except Exception:
    print("the file is at /content/" + base + ".zip")

---
### Reading three of the numbers

**Buried area** is computed pairwise in isolation and reported halved (the interface area,
not the sum of both footprints): PISA's convention, so directly comparable to it.

**Shape complementarity** reads about **0.10 below** the original `sc` program (trypsin–BPTI
0.647; obligate histone heterodimers 0.598–0.620; published ≈ 0.70), so the class names on the
figure are pessimistic. Use S_c to rank interfaces within one run.

**ΔG** is the Eisenberg–McLachlan solvation term (20 amino acids only). Negative = hydrophobic
burial pays; positive = the interface is held by hydrogen bonds and salt bridges instead. It is
not a binding free energy.

### If something goes wrong
Look at the run report (section 8): each item says *ok*, *DEFAULT*, *WARNING*, *SKIPPED* or
*FAILED*, with the reason. A failure in one figure never stops the others. Very large
assemblies may exceed the free Colab machine's memory — run them locally or on a cluster; the
code is identical.
